In [5]:
import numpy as np
import pandas as pd
def build_tableau(A, b):
    m, n = A.shape
    I = np.eye(m)
    tableau = np.hstack([A.astype(float), I, b.reshape(-1,1).astype(float)])
    last_row = np.zeros((1, tableau.shape[1]))
    tableau = np.vstack([tableau, last_row])
    return tableau

def print_tableau(tableau, var_names, basis):
    cols = var_names + ['RHS']
    row_labels = basis + ['ObjRow']
    df = pd.DataFrame(np.round(tableau, 6), index=row_labels, columns=cols)
    print(df)
    print()

def compute_reduced(tableau, full_c, var_names, basis):
    m = tableau.shape[0] - 1
    n = len(var_names)
    N = tableau[:m, :n]       # coefficient matrix including slack cols
    b = tableau[:m, -1]       # RHS
    # Basis indices
    basis_indices = [var_names.index(bv) for bv in basis]
    B = N[:, basis_indices]
    # invert B
    B_inv = np.linalg.inv(B)
    # C_B vector 
    C_B = np.array([full_c[var_names.index(bv)] for bv in basis])
    # z_j = C_B^T * B_inv * N
    z = C_B.dot(B_inv.dot(N))
    reduced = full_c - z
    obj_val = float(C_B.dot(B_inv.dot(b)))
    return reduced, obj_val

def entering(reduced):
    #Pick entering variable column#
    max_val = np.max(reduced)
    if max_val <= 1e-9:
        return None
    return int(np.argmax(reduced))

def leaving(tableau, enter_col):
   #Ratio test
    m = tableau.shape[0] - 1
    col_vals = tableau[:m, enter_col]
    rhs = tableau[:m, -1]
    ratios = []
    for i in range(m):
        a = col_vals[i]
        if a > 1e-9:
            ratios.append(rhs[i] / a)
        else:
            ratios.append(np.inf)
    min_ratio = min(ratios)
    if min_ratio == np.inf:
        return None
    return int(np.argmin(ratios))

def pivot(tableau, row, col):
    #Perform pivot
    pivot_val = tableau[row, col]
    tableau[row, :] = tableau[row, :] / pivot_val
    m, n = tableau.shape
    for r in range(m):
        if r != row:
            tableau[r, :] = tableau[r, :] - tableau[r, col] * tableau[row, :]
    return tableau

def simplex(A, b, c):
    m, n = A.shape
    # Build initial tableau and variable names
    tableau = build_tableau(A, b)
    var_names = [f"x{i+1}" for i in range(n)] + [f"s{i+1}" for i in range(m)]
    full_c = np.hstack([c, np.zeros(m)])   # cost vector for x and slack vars
    basis = [f"s{i+1}" for i in range(m)]  # initial basis = slack variables
    steps = []

    while True:
        reduced, obj_val = compute_reduced(tableau, full_c, var_names, basis)
        tableau[-1, :len(var_names)] = reduced
        tableau[-1, -1] = obj_val
        steps.append((tableau.copy(), list(basis)))
        enter = entering(reduced)
        if enter is None:               # optimal
            break
        leave = leaving(tableau, enter)
        if leave is None:
            raise RuntimeError("Unbounded problem (no valid leaving variable).")
        # update basis and pivot
        basis[leave] = var_names[enter]
        tableau = pivot(tableau, leave, enter)

    return steps, var_names

if __name__ == "__main__":

    # Original minimization 
    A = np.array([
        [1.0, 8.0/3.0],
        [1.0, 1.0],
        [2.0, 0.0]
    ])
    b = np.array([4.0, 2.0, 3.0])
    c = np.array([2.0, 1.0])   # maximize z = 2 x1 + 1 x2

    # Run simplex
    steps, var_names = simplex(A, b, c)

    # Print step-by-step tableaux
    for i, (tab, basis) in enumerate(steps):
        print(f"--- Step {i}  (basis = {basis}) ---")
        print_tableau(tab, var_names, basis)
        
    final_tab, final_basis = steps[-1]
    m = final_tab.shape[0] - 1
    solution = {vn: 0.0 for vn in var_names}
    for i, bv in enumerate(final_basis):
        solution[bv] = final_tab[i, -1]
    x_solution = {k: solution[k] for k in solution if k.startswith('x')}
    z_val = final_tab[-1, -1]

    print("Optimal solution (maximization form):")
    print(x_solution)
    print(f"z_max = {z_val:.6f}")
    print(f"For original minimization f = -z, f_min = {-z_val:.6f}")


--- Step 0  (basis = ['s1', 's2', 's3']) ---
         x1        x2   s1   s2   s3  RHS
s1      1.0  2.666667  1.0  0.0  0.0  4.0
s2      1.0  1.000000  0.0  1.0  0.0  2.0
s3      2.0  0.000000  0.0  0.0  1.0  3.0
ObjRow  2.0  1.000000  0.0  0.0  0.0  0.0

--- Step 1  (basis = ['s1', 's2', 'x1']) ---
         x1        x2   s1   s2   s3  RHS
s1      0.0  2.666667  1.0  0.0 -0.5  2.5
s2      0.0  1.000000  0.0  1.0 -0.5  0.5
x1      1.0  0.000000  0.0  0.0  0.5  1.5
ObjRow  0.0  1.000000  0.0  0.0 -1.0  3.0

--- Step 2  (basis = ['s1', 'x2', 'x1']) ---
         x1   x2   s1        s2        s3       RHS
s1      0.0  0.0  1.0 -2.666667  0.833333  1.166667
x2      0.0  1.0  0.0  1.000000 -0.500000  0.500000
x1      1.0  0.0  0.0  0.000000  0.500000  1.500000
ObjRow  0.0  0.0  0.0 -1.000000 -0.500000  3.500000

Optimal solution (maximization form):
{'x1': np.float64(1.5), 'x2': np.float64(0.5)}
z_max = 3.500000
For original minimization f = -z, f_min = -3.500000
